# MobileNetV2 + TSM — Temporal-Order Held-Out Evaluation
## RWF-2000 + UCF-Crime Fighting + Bus Violence
### TEST ONLY / STANDALONE

Notebook ini dibuat dari protokol executable notebook:

`Seed123_MobileNetTSM_TemporalOrder_TestOnly(1).ipynb`

dan memperluas evaluasi ke tiga held-out dataset:

1. **RWF-2000**
2. **UCF-Crime Fighting**
3. **Bus Violence**

Ketiga dataset selalu dievaluasi **terpisah**. Tidak ada pooling antar-held-out
dataset.

Protokol temporal-order tetap sama dengan notebook sumber:

```text
shared 16-frame cache
→ deterministic evaluation selection:
  [1,3,5,7,9,11,13,15]
→ T = 8
→ Original / Reverse / chunk-wise Shuffle
→ MobileNetV2 + Zhang Method-2 TSM
→ 10 inserted TSM blocks
→ shift_div = 8
→ per-segment logits
→ mean-logit consensus
→ clip prediction
```

Perturbation levels pada T=8:

```text
Shuffle 25%  → chunk_size = 2
Shuffle 50%  → chunk_size = 4
Shuffle 100% → chunk_size = 8
```

Setiap shuffle dijalankan **10 repetitions**.

Tidak ada frame yang ditambah, dihapus, atau diganti. Hanya urutan dari
delapan selected frames yang diubah.

## Mengganti seed

Untuk mengevaluasi checkpoint seed lain, **cukup ganti satu variabel**:

```python
RUN_SEED = 0
```

menjadi `42` atau `123`.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import gc
import json
import random
import platform
import warnings
import zlib

from contextlib import nullcontext
from pathlib import Path
from typing import (
    Dict,
    List,
    Optional,
    Tuple,
)

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

import torchvision

from torchvision.models import (
    mobilenet_v2,
)

from torchvision.models.mobilenetv2 import (
    InvertedResidual,
)

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("CUDA        :", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU         :",
        torch.cuda.get_device_name(0),
    )

Python      : 3.13.15
PyTorch     : 2.11.0+cu128
torchvision : 0.26.0+cu128
CUDA        : True
GPU         : NVIDIA A100-SXM4-40GB


## 1. Configuration

### Hanya ini yang diubah antar-seed

```python
RUN_SEED = 0
```

Valid:

```python
0, 42, 123
```

Konfigurasi berikut dikunci agar eksperimen antar-seed tetap controlled:

```python
CACHE_SEQUENCE_LENGTH = 16
NUM_SEGMENTS = 8
TEST_SEGMENT_INDICES = [1,3,5,7,9,11,13,15]

SHIFT_DIV = 8
EXPECTED_TSM_BLOCKS = 10

SHUFFLE_CHUNK_SIZES = [2,4,8]
NUM_RANDOM_REPEATS = 10
PERMUTATION_BASE_SEED = 20260810

BATCH_SIZE = 8
USE_AMP = False
```

In [ ]:
# ============================================================
# SATU-SATUNYA VARIABEL YANG DIUBAH ANTAR-SEED
# ============================================================
RUN_SEED = 0

VALID_SEEDS = (
    0,
    42,
    123,
)

if RUN_SEED not in VALID_SEEDS:
    raise ValueError(
        f"RUN_SEED harus salah satu dari {VALID_SEEDS}; "
        f"mendapat {RUN_SEED}."
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# ============================================================
# EXACT MODEL / DATA SETTINGS FROM SOURCE NOTEBOOK
# ============================================================
MODEL_VARIANT = "mobilenet_tsm"

USE_TSM = True

CLASS_NAMES = [
    "NonViolence",
    "Violence",
]

NUM_CLASSES = 2

CACHE_SEQUENCE_LENGTH = 16

RAW_HEIGHT = 64
RAW_WIDTH = 64

NUM_SEGMENTS = 8

SEGMENT_WIDTH = (
    CACHE_SEQUENCE_LENGTH
    // NUM_SEGMENTS
)

BACKBONE_INPUT_SIZE = 224

SHIFT_DIV = 8

EXPECTED_TSM_BLOCKS = 10

TEST_SEGMENT_INDICES = np.asarray(
    [
        1,
        3,
        5,
        7,
        9,
        11,
        13,
        15,
    ],
    dtype=np.int64,
)

PREFER_LEGACY_CACHE = True

LEGACY_CACHE_COLOR_ORDER = "BGR"

AUTO_SCALE_LEGACY_CACHE = True

ALLOW_BUILD_CACHE = True

# ============================================================
# EXACT TEMPORAL-ORDER PROTOCOL FROM SOURCE NOTEBOOK
# ============================================================
SHUFFLE_CHUNK_SIZES = [
    2,
    4,
    8,
]

NUM_RANDOM_REPEATS = 10

# Permutation seed only; deliberately independent of checkpoint seed.
PERMUTATION_BASE_SEED = 20260810

RUN_REVERSE = True

# ============================================================
# EVALUATION
# ============================================================
BATCH_SIZE = 8

NUM_WORKERS = 2

# Source comparator uses USE_AMP=False.
USE_AMP = False

# Identity route:
# direct T8 and temporal-order Original must match.
ORIGINAL_PROBABILITY_ATOL = 1e-6

ORIGINAL_METRIC_ATOL = 1e-10

# Optional reproduction of the source notebook's saved-test validity check.
# 0.001 absolute score = 0.1 percentage point.
STRICT_SAVED_RWF_MATCH = True

MAX_ALLOWED_SAVED_RWF_DELTA = 1e-3

# ============================================================
# PATHS
# ============================================================
MODELS_ROOT = Path(
    "/content/drive/MyDrive/Models"
)

COMPARATOR_ROOT = (
    MODELS_ROOT
    / "Comparators"
)

EFFTEMPNET_ROOT = (
    MODELS_ROOT
    / "EffTempNet"
)

BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/Benchmark"
)

CHECKPOINT_PATH_OVERRIDE = None


def resolve_checkpoint(
    seed: int,
) -> Path:
    if CHECKPOINT_PATH_OVERRIDE is not None:
        path = Path(
            CHECKPOINT_PATH_OVERRIDE
        )

        if not path.exists():
            raise FileNotFoundError(
                f"CHECKPOINT_PATH_OVERRIDE tidak ditemukan: {path}"
            )

        return path

    # Exact primary layout from uploaded source notebook:
    # Comparators/mobilenet_tsm_SeedX_v1/mobilenet_tsm/seed_X/best.pt
    candidates = [
        (
            COMPARATOR_ROOT
            / f"mobilenet_tsm_Seed{seed}_v1"
            / MODEL_VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
        (
            COMPARATOR_ROOT
            / f"Zhang2022_mobilenet_tsm_Seed{seed}_v1"
            / MODEL_VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
    ]

    matches = [
        path
        for path in candidates
        if path.exists()
    ]

    matches = list(
        dict.fromkeys(matches)
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        raise RuntimeError(
            "Ditemukan beberapa checkpoint MobileNetV2+TSM T=8. "
            "Set CHECKPOINT_PATH_OVERRIDE:\n"
            + "\n".join(
                str(path)
                for path in matches
            )
        )

    recursive = []

    for path in COMPARATOR_ROOT.glob(
        f"**/{MODEL_VARIANT}/seed_{seed}/best.pt"
    ):
        low = str(path).lower()

        # Exclude native-T retraining variants. Temporal-order source uses
        # the legacy/fixed T=8 checkpoint.
        if (
            "native_t" in low
            or "nativet" in low
        ):
            continue

        recursive.append(path)

    recursive = sorted(
        dict.fromkeys(recursive)
    )

    if len(recursive) == 1:
        return recursive[0]

    if len(recursive) == 0:
        raise FileNotFoundError(
            "Checkpoint MobileNetV2+TSM fixed T=8 tidak ditemukan.\n"
            f"Seed={seed}\n"
            f"Search root={COMPARATOR_ROOT}\n"
            "Jika naming folder berbeda, isi CHECKPOINT_PATH_OVERRIDE."
        )

    raise RuntimeError(
        "Ditemukan beberapa checkpoint MobileNetV2+TSM yang ambigu. "
        "Set CHECKPOINT_PATH_OVERRIDE:\n"
        + "\n".join(
            str(path)
            for path in recursive
        )
    )


CHECKPOINT_PATH = resolve_checkpoint(
    RUN_SEED
)

RUN_DIR = CHECKPOINT_PATH.parent

SAVED_TEST_METRICS_PATH = (
    RUN_DIR
    / "test_metrics.csv"
)

# ============================================================
# HELD-OUT DATASETS
# ============================================================
EXISTING_CACHE_DIR = (
    EFFTEMPNET_ROOT
    / "new_saved_dataset"
)

HELDOUT_CACHE_DIR = (
    EFFTEMPNET_ROOT
    / "heldout_cache_rgb64_v2"
)

HELDOUT_SPECS = {
    "RWF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "RWF"
        ),
        "cache_name": "RWF",
        "cache_dir": EXISTING_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "UCF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "UCF"
        ),
        "cache_name": "UCF",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "BV": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "BV"
        ),
        "cache_name": "BV",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
}

NONVIOLENCE_ALIASES = (
    "NonViolence",
    "Non-Violence",
    "Non_Violence",
    "Nonviolent",
    "NonViolent",
    "NonFight",
    "Non-Fight",
    "Non_Fight",
    "Normal",
    "NormalVideos",
    "Normal_Videos",
)

VIOLENCE_ALIASES = (
    "Violence",
    "Violent",
    "Fight",
    "Fighting",
    "FightVideos",
    "ViolentVideos",
    "Abnormal",
)

EVAL_OUTPUT_DIR = (
    EFFTEMPNET_ROOT
    / "HeldOut_Evaluation"
    / "MobileNetV2_TSM_TemporalOrder"
    / f"seed_{RUN_SEED}"
)

EVAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HELDOUT_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# FIXED PROTOCOL AUDITS
# ============================================================
if (
    CACHE_SEQUENCE_LENGTH
    % NUM_SEGMENTS
    != 0
):
    raise RuntimeError(
        "Cache 16-frame harus dapat dibagi menjadi 8 segments."
    )

expected_indices = np.asarray(
    [
        segment_id
        * SEGMENT_WIDTH
        + SEGMENT_WIDTH
        // 2
        for segment_id in range(
            NUM_SEGMENTS
        )
    ],
    dtype=np.int64,
)

if not np.array_equal(
    TEST_SEGMENT_INDICES,
    expected_indices,
):
    raise RuntimeError(
        "TEST_SEGMENT_INDICES tidak cocok dengan source comparator."
    )

for chunk_size in SHUFFLE_CHUNK_SIZES:
    if (
        NUM_SEGMENTS
        % chunk_size
        != 0
    ):
        raise RuntimeError(
            f"T={NUM_SEGMENTS} tidak habis dibagi chunk={chunk_size}."
        )

print("RUN_SEED          :", RUN_SEED)
print("DEVICE            :", DEVICE)
print("CHECKPOINT        :", CHECKPOINT_PATH)
print("SAVED TEST METRIC :", SAVED_TEST_METRICS_PATH)
print("T=8 indices       :", TEST_SEGMENT_INDICES.tolist())
print("Shuffle chunks    :", SHUFFLE_CHUNK_SIZES)
print("Shuffle repeats   :", NUM_RANDOM_REPEATS)
print("Permutation seed  :", PERMUTATION_BASE_SEED)
print("OUTPUT            :", EVAL_OUTPUT_DIR)

RUN_SEED          : 0
DEVICE            : cuda
CHECKPOINT        : /content/drive/MyDrive/IVS/ViolenceDetection/Models/Comparators/mobilenet_tsm_Seed0_v1/mobilenet_tsm/seed_0/best.pt
SAVED TEST METRIC : /content/drive/MyDrive/IVS/ViolenceDetection/Models/Comparators/mobilenet_tsm_Seed0_v1/mobilenet_tsm/seed_0/test_metrics.csv
T=8 indices       : [1, 3, 5, 7, 9, 11, 13, 15]
Shuffle chunks    : [2, 4, 8]
Shuffle repeats   : 10
Permutation seed  : 20260810
OUTPUT            : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_TemporalOrder/seed_0


In [4]:
AMP_ENABLED = bool(
    USE_AMP
    and DEVICE.type
    == "cuda"
)

AMP_DTYPE = (
    torch.bfloat16
    if (
        AMP_ENABLED
        and torch.cuda.is_bf16_supported()
    )
    else torch.float16
)


def amp_context():
    if not AMP_ENABLED:
        return nullcontext()

    return torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=True,
    )


def set_eval_seed(
    seed: int,
):
    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(
    worker_id: int,
):
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def safe_torch_load(
    path: Path,
    map_location,
):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


def assert_finite_tensor(
    name: str,
    tensor: torch.Tensor,
):
    if not torch.isfinite(
        tensor
    ).all():
        raise RuntimeError(
            f"{name} mengandung NaN/Inf."
        )


set_eval_seed(
    RUN_SEED
)

print(
    "AMP enabled:",
    AMP_ENABLED,
)

AMP enabled: False


## 2. Exact MobileNetV2 + TSM model

Model definition mengikuti notebook sumber:

```text
MobileNetV2
→ TSM inserted after expansion layer
  pada eligible residual InvertedResidual blocks
→ expected TSM blocks = 10
→ shift_div = 8
→ per-segment logits
→ mean over 8 segment logits
```

Checkpoint dimuat menggunakan `strict=True`.

In [5]:
class TemporalShift(
    nn.Module
):
    def __init__(
        self,
        n_segment: int = NUM_SEGMENTS,
        n_div: int = SHIFT_DIV,
    ):
        super().__init__()

        self.n_segment = int(
            n_segment
        )

        self.n_div = int(
            n_div
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:

        nt, c, h, w = (
            x.shape
        )

        if (
            nt
            % self.n_segment
            != 0
        ):
            raise ValueError(
                f"B*T={nt} tidak habis dibagi "
                f"T={self.n_segment}."
            )

        batch_size = (
            nt
            // self.n_segment
        )

        x = x.view(
            batch_size,
            self.n_segment,
            c,
            h,
            w,
        )

        fold = (
            c
            // self.n_div
        )

        if fold == 0:
            return x.view(
                nt,
                c,
                h,
                w,
            )

        out = torch.zeros_like(
            x
        )

        out[
            :,
            :-1,
            :fold,
        ] = x[
            :,
            1:,
            :fold,
        ]

        out[
            :,
            1:,
            fold:2 * fold,
        ] = x[
            :,
            :-1,
            fold:2 * fold,
        ]

        out[
            :,
            :,
            2 * fold:,
        ] = x[
            :,
            :,
            2 * fold:,
        ]

        return out.view(
            nt,
            c,
            h,
            w,
        )

def insert_zhang_method2_tsm(
    model: nn.Module,
    n_segment: int = NUM_SEGMENTS,
    n_div: int = SHIFT_DIV,
) -> int:

    inserted = 0

    for block in model.features:

        if not isinstance(
            block,
            InvertedResidual,
        ):
            continue

        if not block.use_res_connect:
            continue

        if (
            len(
                block.conv
            )
            < 4
        ):
            continue

        expansion = (
            block.conv[
                0
            ]
        )

        block.conv[
            0
        ] = nn.Sequential(
            expansion,
            TemporalShift(
                n_segment=n_segment,
                n_div=n_div,
            ),
        )

        inserted += 1

    return inserted

class VideoMobileNetComparator(
    nn.Module
):
    def __init__(
        self,
        use_tsm: bool = True,
        pretrained: bool = False,
        num_classes: int = NUM_CLASSES,
    ):
        super().__init__()

        self.use_tsm = bool(
            use_tsm
        )

        self.num_segments = (
            NUM_SEGMENTS
        )

        # pretrained=False because trained checkpoint will be loaded.
        self.base_model = (
            mobilenet_v2(
                weights=None
            )
        )

        in_features = (
            self.base_model
            .classifier[1]
            .in_features
        )

        self.base_model.classifier[
            1
        ] = nn.Linear(
            in_features,
            num_classes,
        )

        self.tsm_blocks = 0

        if self.use_tsm:
            self.tsm_blocks = (
                insert_zhang_method2_tsm(
                    self.base_model,
                    n_segment=NUM_SEGMENTS,
                    n_div=SHIFT_DIV,
                )
            )

    def forward(
        self,
        clip: torch.Tensor,
    ) -> torch.Tensor:

        if (
            clip.ndim
            != 5
        ):
            raise ValueError(
                f"Expected [B,T,C,H,W], "
                f"got {tuple(clip.shape)}"
            )

        (
            batch_size,
            seq_len,
            channels,
            height,
            width,
        ) = clip.shape

        if (
            seq_len
            != self.num_segments
        ):
            raise ValueError(
                f"Expected {self.num_segments} segments, "
                f"got {seq_len}"
            )

        x = clip.reshape(
            batch_size
            * seq_len,
            channels,
            height,
            width,
        )

        x = F.interpolate(
            x,
            size=(
                BACKBONE_INPUT_SIZE,
                BACKBONE_INPUT_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        segment_logits = (
            self.base_model(
                x
            )
        )

        segment_logits = (
            segment_logits.view(
                batch_size,
                seq_len,
                NUM_CLASSES,
            )
        )

        return segment_logits.mean(
            dim=1
        )

In [6]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}"
    )

checkpoint = safe_torch_load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if not isinstance(
    checkpoint,
    dict,
):
    raise TypeError(
        "Checkpoint bukan dictionary."
    )

if (
    "model_state_dict"
    not in checkpoint
):
    raise KeyError(
        "Checkpoint tidak memiliki model_state_dict."
    )

checkpoint_config = checkpoint.get(
    "config",
    {},
)

if "seed" in checkpoint_config:
    if int(
        checkpoint_config[
            "seed"
        ]
    ) != RUN_SEED:
        raise RuntimeError(
            "Checkpoint seed mismatch."
        )

if "use_tsm" in checkpoint_config:
    if not bool(
        checkpoint_config[
            "use_tsm"
        ]
    ):
        raise RuntimeError(
            "Checkpoint metadata menyatakan use_tsm=False."
        )

if "num_segments" in checkpoint_config:
    if int(
        checkpoint_config[
            "num_segments"
        ]
    ) != NUM_SEGMENTS:
        raise RuntimeError(
            "Checkpoint metadata bukan T=8."
        )

if "shift_div" in checkpoint_config:
    if int(
        checkpoint_config[
            "shift_div"
        ]
    ) != SHIFT_DIV:
        raise RuntimeError(
            "Checkpoint shift_div mismatch."
        )

model = VideoMobileNetComparator(
    use_tsm=True,
    pretrained=False,
).to(
    DEVICE
)

if (
    model.tsm_blocks
    != EXPECTED_TSM_BLOCKS
):
    raise RuntimeError(
        f"Expected {EXPECTED_TSM_BLOCKS} inserted TSM blocks, "
        f"got {model.tsm_blocks}."
    )

load_result = model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(
        False
    )

trainable_after_freeze = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

if trainable_after_freeze != 0:
    raise RuntimeError(
        "Masih ada trainable parameters."
    )

runtime_tsm_modules = [
    module
    for module in model.modules()
    if isinstance(
        module,
        TemporalShift,
    )
]

if len(
    runtime_tsm_modules
) != EXPECTED_TSM_BLOCKS:
    raise RuntimeError(
        f"Expected {EXPECTED_TSM_BLOCKS} TemporalShift modules, "
        f"got {len(runtime_tsm_modules)}."
    )

with torch.inference_mode():
    dummy = torch.zeros(
        2,
        NUM_SEGMENTS,
        3,
        RAW_HEIGHT,
        RAW_WIDTH,
        device=DEVICE,
    )

    dummy_logits = model(
        dummy
    )

if tuple(
    dummy_logits.shape
) != (
    2,
    NUM_CLASSES,
):
    raise RuntimeError(
        f"Unexpected output shape: {tuple(dummy_logits.shape)}"
    )

assert_finite_tensor(
    "dummy_logits",
    dummy_logits,
)

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("✅ Strict MobileNetV2+TSM checkpoint/model audit PASS")
print("Epoch            :", checkpoint.get("epoch", "unknown"))
print("Best score       :", checkpoint.get("best_score", "unknown"))
print("TSM blocks       :", model.tsm_blocks)
print("Segments         :", model.num_segments)
print("Shift div        :", SHIFT_DIV)
print("Parameters       :", f"{total_params:,}")
print("Trainable params :", trainable_after_freeze)
print("Load result      :", load_result)

del dummy, dummy_logits

if torch.cuda.is_available():
    torch.cuda.empty_cache()

✅ Strict MobileNetV2+TSM checkpoint/model audit PASS
Epoch            : 63
Best score       : 0.9504283965728274
TSM blocks       : 10
Segments         : 8
Shift div        : 8
Parameters       : 2,226,434
Trainable params : 0
Load result      : <All keys matched successfully>


## 3. Resolve / build held-out 16-frame cache

RWF memakai cache eksperimen yang sudah ada jika tersedia.

UCF dan Bus Violence memakai shared held-out cache yang sama dengan evaluator
standalone sebelumnya. Jika cache UCF/BV belum ada dan raw folder tersedia,
notebook dapat membuat RGB-v2 cache.

Cache-builder dibuat **self-contained** agar tidak bergantung pada
`SEQUENCE_LENGTH` dari cell lain.

In [7]:
# ============================================================
# SELF-CONTAINED CACHE-BUILDER CONSTANTS
# ============================================================
_CACHE_BUILDER_SEQUENCE_LENGTH = 16
_CACHE_BUILDER_RAW_HEIGHT = 64
_CACHE_BUILDER_RAW_WIDTH = 64

if (
    "CACHE_SEQUENCE_LENGTH" in globals()
    and int(
        globals()[
            "CACHE_SEQUENCE_LENGTH"
        ]
    )
    != _CACHE_BUILDER_SEQUENCE_LENGTH
):
    raise RuntimeError(
        "Protocol mismatch: CACHE_SEQUENCE_LENGTH harus 16."
    )

if (
    "RAW_HEIGHT" in globals()
    and int(
        globals()[
            "RAW_HEIGHT"
        ]
    )
    != _CACHE_BUILDER_RAW_HEIGHT
):
    raise RuntimeError(
        "Protocol mismatch: RAW_HEIGHT harus 64."
    )

if (
    "RAW_WIDTH" in globals()
    and int(
        globals()[
            "RAW_WIDTH"
        ]
    )
    != _CACHE_BUILDER_RAW_WIDTH
):
    raise RuntimeError(
        "Protocol mismatch: RAW_WIDTH harus 64."
    )

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm"
}

def direct_subdirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {
        p.name: p
        for p in root.iterdir()
        if p.is_dir()
    }

def resolve_one_class_dir(
    root: Path,
    explicit_name: Optional[str],
    aliases: Tuple[str, ...],
    canonical_name: str,
) -> Path:
    if explicit_name is not None:
        candidate = root / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(
                f"Folder {canonical_name} tidak ditemukan: {candidate}"
            )
        return candidate

    subdirs = direct_subdirs(root)
    lower_map = {
        name.lower(): path
        for name, path in subdirs.items()
    }

    matches = []
    for alias in aliases:
        candidate = lower_map.get(alias.lower())
        if candidate is not None and candidate not in matches:
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Tidak dapat menentukan folder kelas {canonical_name} "
            f"secara aman pada {root}.\n"
            f"Matches: {[p.name for p in matches]}\n"
            f"Available: {sorted(subdirs.keys())}\n"
            "Isi nonviolence_dir/violence_dir di HELDOUT_SPECS."
        )

    return matches[0]

def list_videos_for_spec(
    raw_dir: Path,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> List[Tuple[Path, int]]:
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Raw held-out directory tidak ditemukan: {raw_dir}"
        )

    class_dirs = {
        0: resolve_one_class_dir(
            raw_dir,
            nonviolence_dir,
            NONVIOLENCE_ALIASES,
            "NonViolence",
        ),
        1: resolve_one_class_dir(
            raw_dir,
            violence_dir,
            VIOLENCE_ALIASES,
            "Violence",
        ),
    }

    records = []

    for label, class_dir in class_dirs.items():
        videos = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )

        if not videos:
            raise RuntimeError(
                f"Tidak ada video pada folder kelas: {class_dir}"
            )

        print(
            f"{CLASS_NAMES[label]:12s}: {len(videos):5d} clips | {class_dir}"
        )

        records.extend(
            (p, label)
            for p in videos
        )

    labels = np.asarray(
        [label for _, label in records],
        dtype=np.int64,
    )

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise RuntimeError(
            "Held-out dataset harus mempunyai kedua kelas: "
            "NonViolence dan Violence."
        )

    return records

def extract_uniform_rgb_frames(
    video_path: Path,
    sequence_length: int = _CACHE_BUILDER_SEQUENCE_LENGTH,
    height: int = _CACHE_BUILDER_RAW_HEIGHT,
    width: int = _CACHE_BUILDER_RAW_WIDTH,
) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if frame_count <= 0:
        cap.release()
        return None

    frame_positions = np.linspace(
        0,
        max(frame_count - 1, 0),
        sequence_length,
    ).astype(int)

    frames = []

    for pos in frame_positions:
        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(pos),
        )

        ok, frame_bgr = cap.read()

        if not ok:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        frame_rgb = cv2.resize(
            frame_rgb,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )

        frames.append(
            frame_rgb.astype(np.float32) / 255.0
        )

    cap.release()

    if not frames:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.stack(
        frames[:sequence_length],
        axis=0,
    ).astype(np.float32)

def build_rgb_cache(
    raw_dir: Path,
    cache_dir: Path,
    cache_name: str,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_path = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    label_path = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    paths_path = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    records = list_videos_for_spec(
        raw_dir,
        nonviolence_dir,
        violence_dir,
    )

    # Gunakan temporary memmap agar tidak menyimpan semua clip di RAM.
    n_requested = len(records)

    temp_path = cache_dir / f".{cache_name}_tmp_features.npy"

    if temp_path.exists():
        temp_path.unlink()

    temp = np.lib.format.open_memmap(
        temp_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_requested,
            _CACHE_BUILDER_SEQUENCE_LENGTH,
            _CACHE_BUILDER_RAW_HEIGHT,
            _CACHE_BUILDER_RAW_WIDTH,
            3,
        ),
    )

    labels_buffer = np.empty(
        n_requested,
        dtype=np.int64,
    )

    valid_paths = []
    failed_paths = []
    valid_count = 0

    for video_path, label in tqdm(
        records,
        desc=f"Build cache {cache_name}",
        unit="video",
    ):
        clip = extract_uniform_rgb_frames(video_path)

        if clip is None:
            failed_paths.append(str(video_path))
            continue

        temp[valid_count] = clip
        labels_buffer[valid_count] = int(label)
        valid_paths.append(str(video_path))
        valid_count += 1

    temp.flush()
    del temp

    if valid_count == 0:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(
            f"Semua video gagal didekode untuk {cache_name}."
        )

    # Compact hanya sample valid ke final .npy.
    temp_read = np.load(
        temp_path,
        mmap_mode="r",
    )

    final = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            valid_count,
            _CACHE_BUILDER_SEQUENCE_LENGTH,
            _CACHE_BUILDER_RAW_HEIGHT,
            _CACHE_BUILDER_RAW_WIDTH,
            3,
        ),
    )

    chunk_size = 16

    for start in range(0, valid_count, chunk_size):
        end = min(start + chunk_size, valid_count)
        final[start:end] = temp_read[start:end]

    final.flush()
    del final, temp_read

    np.save(
        label_path,
        labels_buffer[:valid_count],
    )

    with open(
        paths_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "paths": valid_paths,
                "failed": failed_paths,
                "n_requested": n_requested,
                "n_valid": valid_count,
                "color_order": "RGB",
                "sequence_length": _CACHE_BUILDER_SEQUENCE_LENGTH,
                "height": _CACHE_BUILDER_RAW_HEIGHT,
                "width": _CACHE_BUILDER_RAW_WIDTH,
            },
            f,
            indent=2,
        )

    if temp_path.exists():
        temp_path.unlink()

    print(
        f"Saved {feature_path} | "
        f"shape=({valid_count},{_CACHE_BUILDER_SEQUENCE_LENGTH},{_CACHE_BUILDER_RAW_HEIGHT},{_CACHE_BUILDER_RAW_WIDTH},3)"
    )
    print("Failed videos:", len(failed_paths))

    return {
        "feature_path": feature_path,
        "label_path": label_path,
        "paths_path": paths_path,
        "color_order": "RGB",
        "cache_source": "built_rgb64_v2",
    }

def resolve_heldout_store(
    dataset_name: str,
    spec: Dict,
) -> Dict:
    raw_dir = Path(spec["raw_dir"])
    cache_dir = Path(spec["cache_dir"])
    cache_name = str(spec["cache_name"])

    rgb_feature = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    rgb_label = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    rgb_paths = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    legacy_feature = cache_dir / f"{cache_name}_features.npy"
    legacy_label = cache_dir / f"{cache_name}_labels.npy"
    legacy_paths = cache_dir / f"{cache_name}_paths.json"

    # Sama dengan training: RGB-v2 diprioritaskan; legacy dipakai jika v2 tidak ada.
    if rgb_feature.exists() and rgb_label.exists():
        print(f"✅ {dataset_name}: RGB-v2 cache")
        return {
            "feature_path": rgb_feature,
            "label_path": rgb_label,
            "paths_path": rgb_paths if rgb_paths.exists() else None,
            "color_order": "RGB",
            "cache_source": "rgb64_v2",
        }

    if (
        PREFER_LEGACY_CACHE
        and legacy_feature.exists()
        and legacy_label.exists()
    ):
        print(
            f"✅ {dataset_name}: legacy cache "
            f"({LEGACY_CACHE_COLOR_ORDER}→RGB saat loading)"
        )
        return {
            "feature_path": legacy_feature,
            "label_path": legacy_label,
            "paths_path": legacy_paths if legacy_paths.exists() else None,
            "color_order": LEGACY_CACHE_COLOR_ORDER,
            "cache_source": "legacy",
        }

    if not ALLOW_BUILD_CACHE:
        raise FileNotFoundError(
            f"Cache {dataset_name} tidak ada dan ALLOW_BUILD_CACHE=False."
        )

    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Cache dan raw folder tidak ditemukan untuk {dataset_name}.\n"
            f"Raw expected: {raw_dir}"
        )

    print(
        f"ℹ️ {dataset_name}: cache belum ada, dibuat dari {raw_dir}"
    )

    return build_rgb_cache(
        raw_dir=raw_dir,
        cache_dir=cache_dir,
        cache_name=cache_name,
        nonviolence_dir=spec.get("nonviolence_dir"),
        violence_dir=spec.get("violence_dir"),
    )

## 4. Temporal-order perturbation

Perturbation dilakukan **setelah** fixed 8-frame evaluation selection:

```text
cache T=16:
[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]

selected T=8:
[1,3,5,7,9,11,13,15]
```

Relative order T=8:

```text
Original: [0,1,2,3,4,5,6,7]
Reverse : [7,6,5,4,3,2,1,0]
```

Chunk shuffle:
- `shuffle_2`: 25%
- `shuffle_4`: 50%
- `shuffle_8`: 100%

In [8]:
IMAGENET_MEAN = torch.tensor(
    [
        0.485,
        0.456,
        0.406,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)

IMAGENET_STD = torch.tensor(
    [
        0.229,
        0.224,
        0.225,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)


def stable_dataset_code(
    dataset_name: str,
) -> int:
    return int(
        zlib.crc32(
            dataset_name.encode(
                "utf-8"
            )
        )
        & 0xFFFFFFFF
    )


def make_temporal_order(
    dataset_name: str,
    source_idx: int,
    mode: str,
    repeat_id: int = 0,
    chunk_size: Optional[int] = None,
) -> np.ndarray:
    base_order = np.arange(
        NUM_SEGMENTS,
        dtype=np.int64,
    )

    if mode == "original":
        return base_order

    if mode == "reverse":
        return base_order[
            ::-1
        ].copy()

    if mode != "shuffle":
        raise ValueError(
            f"Unknown mode={mode}"
        )

    if chunk_size not in SHUFFLE_CHUNK_SIZES:
        raise ValueError(
            f"Invalid chunk_size={chunk_size}"
        )

    # RUN_SEED is deliberately NOT included.
    seed_sequence = np.random.SeedSequence(
        [
            int(
                PERMUTATION_BASE_SEED
            ),
            int(
                repeat_id
            ),
            int(
                stable_dataset_code(
                    dataset_name
                )
            ),
            int(
                source_idx
            ),
            int(
                chunk_size
            ),
        ]
    )

    rng = np.random.default_rng(
        seed_sequence
    )

    order = base_order.copy()

    for start in range(
        0,
        NUM_SEGMENTS,
        chunk_size,
    ):
        stop = (
            start
            + chunk_size
        )

        order[
            start:stop
        ] = rng.permutation(
            order[
                start:stop
            ]
        )

    if not np.array_equal(
        np.sort(
            order
        ),
        base_order,
    ):
        raise RuntimeError(
            "Temporal order bukan complete permutation."
        )

    return order


class HeldOut16CacheBase(
    Dataset
):
    def __init__(
        self,
        dataset_name: str,
        feature_path: Path,
        label_path: Path,
        color_order: str,
    ):
        self.dataset_name = str(
            dataset_name
        )

        self.features = np.load(
            feature_path,
            mmap_mode="r",
        )

        self.labels = np.load(
            label_path,
            mmap_mode="r",
        ).astype(
            np.int64
        )

        self.color_order = str(
            color_order
        ).upper()

        expected_shape = (
            CACHE_SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )

        if (
            self.features.ndim != 5
            or tuple(
                self.features.shape[
                    1:
                ]
            )
            != expected_shape
        ):
            raise ValueError(
                f"{dataset_name}: cache shape={self.features.shape}; "
                f"expected [N,{CACHE_SEQUENCE_LENGTH},"
                f"{RAW_HEIGHT},{RAW_WIDTH},3]."
            )

        if (
            len(
                self.features
            )
            != len(
                self.labels
            )
        ):
            raise ValueError(
                f"{dataset_name}: feature/label length mismatch."
            )

        unique_labels = set(
            np.unique(
                self.labels
            ).tolist()
        )

        if unique_labels != {
            0,
            1,
        }:
            raise ValueError(
                f"{dataset_name}: held-out labels harus {{0,1}}, "
                f"found {sorted(unique_labels)}."
            )

        if self.color_order not in {
            "RGB",
            "BGR",
        }:
            raise ValueError(
                f"Unsupported color_order={self.color_order}"
            )

    def __len__(
        self,
    ):
        return len(
            self.labels
        )

    def load_frames(
        self,
        source_idx: int,
    ) -> Tuple[np.ndarray, int]:
        frames = np.asarray(
            self.features[
                source_idx
            ],
            dtype=np.float32,
        )

        label = int(
            self.labels[
                source_idx
            ]
        )

        if not np.isfinite(
            frames
        ).all():
            raise ValueError(
                f"{self.dataset_name}, idx={source_idx}: NaN/Inf."
            )

        frame_min = float(
            frames.min()
        )

        frame_max = float(
            frames.max()
        )

        if (
            AUTO_SCALE_LEGACY_CACHE
            and frame_min >= 0.0
            and frame_max > 1.5
        ):
            if frame_max <= 255.0 + 1e-3:
                frames = (
                    frames
                    / 255.0
                )

            else:
                raise ValueError(
                    f"{self.dataset_name}: invalid cache range "
                    f"[{frame_min},{frame_max}]."
                )

        if (
            float(
                frames.min()
            )
            < -1e-4
            or float(
                frames.max()
            )
            > 1.0001
        ):
            raise ValueError(
                f"{self.dataset_name}: cache harus berada pada [0,1]."
            )

        if self.color_order == "BGR":
            frames = frames[
                ...,
                [
                    2,
                    1,
                    0,
                ],
            ]

        return (
            frames,
            label,
        )


def selected_to_tensor(
    selected: np.ndarray,
) -> torch.Tensor:
    expected_shape = (
        NUM_SEGMENTS,
        RAW_HEIGHT,
        RAW_WIDTH,
        3,
    )

    if tuple(
        selected.shape
    ) != expected_shape:
        raise RuntimeError(
            f"Selected clip shape={selected.shape}; "
            f"expected={expected_shape}."
        )

    clip = torch.from_numpy(
        np.ascontiguousarray(
            selected
        )
    ).permute(
        0,
        3,
        1,
        2,
    )

    return (
        clip
        - IMAGENET_MEAN
    ) / IMAGENET_STD


class MobileNetTSMDirectT8Dataset(
    HeldOut16CacheBase
):
    def __getitem__(
        self,
        source_idx: int,
    ):
        (
            frames,
            label,
        ) = self.load_frames(
            source_idx
        )

        selected = frames[
            TEST_SEGMENT_INDICES
        ]

        clip = selected_to_tensor(
            selected
        )

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(
                source_idx
            ),
        )


class MobileNetTSMTemporalOrderDataset(
    HeldOut16CacheBase
):
    def __init__(
        self,
        dataset_name: str,
        feature_path: Path,
        label_path: Path,
        color_order: str,
        mode: str,
        repeat_id: int = 0,
        chunk_size: Optional[int] = None,
    ):
        super().__init__(
            dataset_name=dataset_name,
            feature_path=feature_path,
            label_path=label_path,
            color_order=color_order,
        )

        self.mode = str(
            mode
        )

        self.repeat_id = int(
            repeat_id
        )

        self.chunk_size = chunk_size

    def __getitem__(
        self,
        source_idx: int,
    ):
        (
            frames,
            label,
        ) = self.load_frames(
            source_idx
        )

        selected = frames[
            TEST_SEGMENT_INDICES
        ]

        order = make_temporal_order(
            dataset_name=self.dataset_name,
            source_idx=int(
                source_idx
            ),
            mode=self.mode,
            repeat_id=self.repeat_id,
            chunk_size=self.chunk_size,
        )

        selected = selected[
            order
        ]

        clip = selected_to_tensor(
            selected
        )

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(
                source_idx
            ),
            torch.from_numpy(
                order.copy()
            ),
        )


def make_loader(
    dataset: Dataset,
):
    generator = torch.Generator()

    generator.manual_seed(
        PERMUTATION_BASE_SEED
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(
            NUM_WORKERS
            > 0
        ),
    )


def load_paths_if_available(
    paths_path: Optional[Path],
    expected_n: int,
) -> Optional[List[str]]:
    if paths_path is None:
        return None

    path = Path(
        paths_path
    )

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(
            file
        )

    paths = payload.get(
        "paths"
    )

    if not isinstance(
        paths,
        list,
    ):
        return None

    if len(
        paths
    ) != expected_n:
        return None

    return [
        str(path)
        for path in paths
    ]


# ============================================================
# PERMUTATION AUDIT
# ============================================================
audit_rows = []

for mode, chunk_size in [
    ("original", None),
    ("reverse", None),
    ("shuffle", 2),
    ("shuffle", 4),
    ("shuffle", 8),
]:
    repeat_ids = (
        [0]
        if mode != "shuffle"
        else [
            0,
            1,
            9,
        ]
    )

    for repeat_id in repeat_ids:
        first = make_temporal_order(
            dataset_name="RWF",
            source_idx=123,
            mode=mode,
            repeat_id=repeat_id,
            chunk_size=chunk_size,
        )

        second = make_temporal_order(
            dataset_name="RWF",
            source_idx=123,
            mode=mode,
            repeat_id=repeat_id,
            chunk_size=chunk_size,
        )

        if not np.array_equal(
            first,
            second,
        ):
            raise RuntimeError(
                "Permutation generator tidak deterministic."
            )

        audit_rows.append(
            {
                "mode": mode,
                "chunk_size": chunk_size,
                "repeat": repeat_id,
                "order_relative_t8_0based": first.tolist(),
                "cache_indices_after_order_0based": (
                    TEST_SEGMENT_INDICES[
                        first
                    ]
                ).tolist(),
            }
        )

permutation_audit_df = pd.DataFrame(
    audit_rows
)

permutation_audit_df.to_json(
    EVAL_OUTPUT_DIR
    / "temporal_order_permutation_audit_examples.json",
    orient="records",
    indent=2,
)

display(
    permutation_audit_df
)

print(
    "✅ Temporal-order permutation audit PASS"
)

,mode,chunk_size,repeat,order_relative_t8_0based,cache_indices_after_order_0based
0,original,NaN,0,"[0, 1, 2, 3, 4, 5, 6, 7]","[1, 3, 5, 7, 9, 11, 13, 15]"
1,reverse,NaN,0,"[7, 6, 5, 4, 3, 2, 1, 0]","[15, 13, 11, 9, 7, 5, 3, 1]"
2,shuffle,2.0,0,"[1, 0, 2, 3, 5, 4, 6, 7]","[3, 1, 5, 7, 11, 9, 13, 15]"
3,shuffle,2.0,1,"[1, 0, 2, 3, 5, 4, 6, 7]","[3, 1, 5, 7, 11, 9, 13, 15]"
4,shuffle,2.0,9,"[0, 1, 3, 2, 4, 5, 6, 7]","[1, 3, 7, 5, 9, 11, 13, 15]"
5,shuffle,4.0,0,"[3, 1, 2, 0, 6, 5, 7, 4]","[7, 3, 5, 1, 13, 11, 15, 9]"
6,shuffle,4.0,1,"[3, 0, 1, 2, 6, 4, 5, 7]","[7, 1, 3, 5, 13, 9, 11, 15]"
7,shuffle,4.0,9,"[1, 3, 2, 0, 4, 7, 6, 5]","[3, 7, 5, 1, 9, 15, 13, 11]"
8,shuffle,8.0,0,"[1, 0, 2, 3, 7, 4, 5, 6]","[3, 1, 5, 7, 15, 9, 11, 13]"
9,shuffle,8.0,1,"[2, 5, 4, 7, 1, 3, 0, 6]","[5, 11, 9, 15, 3, 7, 1, 13]"


✅ Temporal-order permutation audit PASS


## 5. Resolve and audit RWF, UCF, Bus Violence

In [9]:
heldout_stores = {}

dataset_audit_rows = []

for (
    dataset_name,
    spec,
) in HELDOUT_SPECS.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        "DATASET:",
        dataset_name,
    )

    print(
        "=" * 90
    )

    store = resolve_heldout_store(
        dataset_name,
        spec,
    )

    dataset = MobileNetTSMDirectT8Dataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
    )

    labels = np.asarray(
        dataset.labels,
        dtype=np.int64,
    )

    counts = np.bincount(
        labels,
        minlength=2,
    )

    heldout_stores[
        dataset_name
    ] = store

    dataset_audit_rows.append(
        {
            "dataset": dataset_name,
            "n_total": int(
                len(dataset)
            ),
            "n_nonviolence": int(
                counts[0]
            ),
            "n_violence": int(
                counts[1]
            ),
            "cache_source": store[
                "cache_source"
            ],
            "color_order": store[
                "color_order"
            ],
            "feature_path": str(
                store[
                    "feature_path"
                ]
            ),
            "label_path": str(
                store[
                    "label_path"
                ]
            ),
        }
    )

    print(
        f"✅ {dataset_name}: "
        f"N={len(dataset)}, "
        f"NV={counts[0]}, "
        f"V={counts[1]}, "
        f"cache={store['cache_source']}"
    )

dataset_audit_df = pd.DataFrame(
    dataset_audit_rows
)

dataset_audit_df.to_csv(
    EVAL_OUTPUT_DIR
    / "heldout_dataset_audit.csv",
    index=False,
)

display(
    dataset_audit_df
)


DATASET: RWF
✅ RWF: legacy cache (BGR→RGB saat loading)
✅ RWF: N=1998, NV=1000, V=998, cache=legacy

DATASET: UCF
✅ UCF: RGB-v2 cache
✅ UCF: N=100, NV=50, V=50, cache=rgb64_v2

DATASET: BV
✅ BV: RGB-v2 cache
✅ BV: N=1400, NV=700, V=700, cache=rgb64_v2


,dataset,n_total,n_nonviolence,n_violence,cache_source,color_order,feature_path,label_path
0,RWF,1998,1000,998,legacy,BGR,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
1,UCF,100,50,50,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
2,BV,1400,700,700,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...


## 6. Metrics and per-clip output

Metrics:
- Accuracy
- Balanced Accuracy
- Macro Precision / Recall / F1
- class-wise Precision / Recall / F1
- ROC-AUC
- Average Precision

AP dipakai sebagai metric utama pada summary dan delta.

Prediction CSV menyimpan `sample_key`, permutation, hard prediction,
correctness, dan probabilities.

In [10]:
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
) -> Dict[str, float]:
    (
        precision_cls,
        recall_cls,
        f1_cls,
        support,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[
            0,
            1,
        ],
        zero_division=0,
    )

    result = {
        "n": int(
            len(y_true)
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "precision_nonviolence": float(
            precision_cls[0]
        ),

        "recall_nonviolence": float(
            recall_cls[0]
        ),

        "f1_nonviolence": float(
            f1_cls[0]
        ),

        "precision_violence": float(
            precision_cls[1]
        ),

        "recall_violence": float(
            recall_cls[1]
        ),

        "f1_violence": float(
            f1_cls[1]
        ),
    }

    if len(
        np.unique(
            y_true
        )
    ) == 2:
        result[
            "roc_auc"
        ] = float(
            roc_auc_score(
                y_true,
                p_violence,
            )
        )

        result[
            "average_precision"
        ] = float(
            average_precision_score(
                y_true,
                p_violence,
            )
        )

    else:
        result[
            "roc_auc"
        ] = float("nan")

        result[
            "average_precision"
        ] = float("nan")

    return result


@torch.inference_mode()
def run_direct_loader(
    loader: DataLoader,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []

    for (
        clips,
        labels,
        indices,
    ) in tqdm(
        loader,
        leave=False,
        desc="Direct T8",
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        with amp_context():
            logits = model(
                clips
            )

        assert_finite_tensor(
            "direct_logits",
            logits,
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = probabilities.argmax(
            dim=1
        )

        all_indices.extend(
            np.asarray(
                indices
            ).tolist()
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

    return (
        np.asarray(
            all_indices,
            dtype=np.int64,
        ),
        np.asarray(
            all_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_predictions,
            dtype=np.int64,
        ),
        np.asarray(
            all_probabilities,
            dtype=np.float64,
        ),
    )


@torch.inference_mode()
def run_order_loader(
    loader: DataLoader,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []
    all_orders = []

    for (
        clips,
        labels,
        indices,
        orders,
    ) in tqdm(
        loader,
        leave=False,
        desc="Temporal order",
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        with amp_context():
            logits = model(
                clips
            )

        assert_finite_tensor(
            "temporal_order_logits",
            logits,
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = probabilities.argmax(
            dim=1
        )

        all_indices.extend(
            np.asarray(
                indices
            ).tolist()
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

        all_orders.extend(
            orders.numpy().tolist()
        )

    return (
        np.asarray(
            all_indices,
            dtype=np.int64,
        ),
        np.asarray(
            all_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_predictions,
            dtype=np.int64,
        ),
        np.asarray(
            all_probabilities,
            dtype=np.float64,
        ),
        all_orders,
    )


def resolve_sample_keys(
    dataset_name: str,
    store: Dict,
    indices: np.ndarray,
):
    source_paths = load_paths_if_available(
        store.get(
            "paths_path"
        ),
        expected_n=len(indices),
    )

    if source_paths is None:
        sample_keys = [
            f"{dataset_name}::{int(index)}"
            for index in indices
        ]

        video_paths = [
            ""
            for _ in indices
        ]

    else:
        video_paths = [
            source_paths[
                int(index)
            ]
            for index in indices
        ]

        sample_keys = [
            Path(path).as_posix()
            for path in video_paths
        ]

    return (
        sample_keys,
        video_paths,
    )


def evaluate_direct_condition(
    dataset_name: str,
    store: Dict,
):
    dataset = MobileNetTSMDirectT8Dataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
    )

    loader = make_loader(
        dataset
    )

    (
        indices,
        y_true,
        y_pred,
        probabilities,
    ) = run_direct_loader(
        loader
    )

    (
        sample_keys,
        video_paths,
    ) = resolve_sample_keys(
        dataset_name,
        store,
        indices,
    )

    metrics = compute_metrics(
        y_true,
        y_pred,
        probabilities[
            :,
            1
        ],
    )

    direct_predictions_df = pd.DataFrame(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "scenario": "direct_t8",
            "mode": "direct",
            "chunk_size": np.nan,
            "repeat": 0,
            "sample_index": indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "permutation_0based": json.dumps(
                list(
                    range(
                        NUM_SEGMENTS
                    )
                )
            ),
            "y_true": y_true,
            "y_pred": y_pred,
            "is_correct": (
                y_true
                == y_pred
            ).astype(
                np.int64
            ),
            "p_nonviolence": probabilities[
                :,
                0
            ],
            "p_violence": probabilities[
                :,
                1
            ],
        }
    )

    if direct_predictions_df[
        "sample_key"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate sample_key: {dataset_name} direct."
        )

    return (
        metrics,
        direct_predictions_df,
    )


def scenario_name(
    mode: str,
    chunk_size: Optional[int],
) -> str:
    if mode == "original":
        return "original"

    if mode == "reverse":
        return "reverse"

    return f"shuffle_{chunk_size}"


def evaluate_order_condition(
    dataset_name: str,
    store: Dict,
    mode: str,
    repeat_id: int,
    chunk_size: Optional[int] = None,
):
    scenario = scenario_name(
        mode,
        chunk_size,
    )

    dataset = MobileNetTSMTemporalOrderDataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
        mode=mode,
        repeat_id=repeat_id,
        chunk_size=chunk_size,
    )

    loader = make_loader(
        dataset
    )

    (
        indices,
        y_true,
        y_pred,
        probabilities,
        orders,
    ) = run_order_loader(
        loader
    )

    (
        sample_keys,
        video_paths,
    ) = resolve_sample_keys(
        dataset_name,
        store,
        indices,
    )

    metrics = compute_metrics(
        y_true,
        y_pred,
        probabilities[
            :,
            1
        ],
    )

    condition_predictions_df = pd.DataFrame(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "scenario": scenario,
            "mode": mode,
            "chunk_size": chunk_size,
            "repeat": repeat_id,
            "sample_index": indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "permutation_0based": [
                json.dumps(order)
                for order in orders
            ],
            "y_true": y_true,
            "y_pred": y_pred,
            "is_correct": (
                y_true
                == y_pred
            ).astype(
                np.int64
            ),
            "p_nonviolence": probabilities[
                :,
                0
            ],
            "p_violence": probabilities[
                :,
                1
            ],
        }
    )

    if condition_predictions_df[
        "sample_key"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate sample_key: "
            f"{dataset_name}, {scenario}, repeat={repeat_id}."
        )

    return (
        metrics,
        condition_predictions_df,
    )

## 7. Original validity gates

Dua audit digunakan:

### A. Direct-T8 vs TemporalOrder Original
Harus identik secara praktis clip-by-clip.

### B. Optional saved-RWF reproduction
Jika `test_metrics.csv` dari checkpoint tersedia dan mengandung row RWF,
Original RWF dibandingkan dengan saved evaluation menggunakan toleransi
`0.001` absolute score, sama seperti notebook sumber.

UCF dan Bus Violence tidak memiliki saved checkpoint evaluation lama, sehingga
validity gate untuk keduanya menggunakan direct-T8 vs Original.

In [11]:
direct_metric_rows = []

direct_prediction_frames = []

original_metric_rows = []

original_prediction_frames = []

identity_gate_rows = []

for (
    dataset_name,
    store,
) in heldout_stores.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        f"ORIGINAL IDENTITY GATE | {dataset_name}"
    )

    print(
        "=" * 90
    )

    (
        direct_metrics,
        direct_predictions_df,
    ) = evaluate_direct_condition(
        dataset_name,
        store,
    )

    (
        original_metrics,
        original_predictions_df,
    ) = evaluate_order_condition(
        dataset_name=dataset_name,
        store=store,
        mode="original",
        repeat_id=0,
        chunk_size=None,
    )

    merged = direct_predictions_df[
        [
            "sample_key",
            "y_true",
            "p_nonviolence",
            "p_violence",
        ]
    ].merge(
        original_predictions_df[
            [
                "sample_key",
                "y_true",
                "p_nonviolence",
                "p_violence",
            ]
        ],
        on="sample_key",
        how="inner",
        suffixes=(
            "_direct",
            "_original",
        ),
        validate="one_to_one",
    )

    if len(
        merged
    ) != len(
        direct_predictions_df
    ):
        raise RuntimeError(
            f"{dataset_name}: Direct/Original sample-count mismatch."
        )

    if not np.array_equal(
        merged[
            "y_true_direct"
        ].to_numpy(),
        merged[
            "y_true_original"
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{dataset_name}: Direct/Original label mismatch."
        )

    max_probability_delta = float(
        np.max(
            np.abs(
                merged[
                    [
                        "p_nonviolence_direct",
                        "p_violence_direct",
                    ]
                ].to_numpy()
                -
                merged[
                    [
                        "p_nonviolence_original",
                        "p_violence_original",
                    ]
                ].to_numpy()
            )
        )
    )

    metric_deltas = {}

    for metric_name in [
        "accuracy",
        "balanced_accuracy",
        "f1_macro",
        "roc_auc",
        "average_precision",
    ]:
        metric_deltas[
            metric_name
        ] = float(
            original_metrics[
                metric_name
            ]
            -
            direct_metrics[
                metric_name
            ]
        )

    finite_metric_deltas = [
        abs(value)
        for value in metric_deltas.values()
        if np.isfinite(value)
    ]

    max_metric_delta = (
        float(
            max(
                finite_metric_deltas
            )
        )
        if finite_metric_deltas
        else 0.0
    )

    identity_gate_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "max_abs_probability_delta": max_probability_delta,
            "max_abs_metric_delta": max_metric_delta,
            **{
                f"delta_{key}": value
                for key, value in metric_deltas.items()
            },
        }
    )

    if (
        max_probability_delta
        > ORIGINAL_PROBABILITY_ATOL
    ):
        raise RuntimeError(
            f"{dataset_name}: Direct/Original probability mismatch: "
            f"{max_probability_delta} > {ORIGINAL_PROBABILITY_ATOL}"
        )

    if (
        max_metric_delta
        > ORIGINAL_METRIC_ATOL
    ):
        raise RuntimeError(
            f"{dataset_name}: Direct/Original metric mismatch: "
            f"{max_metric_delta} > {ORIGINAL_METRIC_ATOL}"
        )

    direct_metric_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            **direct_metrics,
        }
    )

    direct_prediction_frames.append(
        direct_predictions_df
    )

    original_metric_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "scenario": "original",
            "mode": "original",
            "chunk_size": np.nan,
            "repeat": 0,
            "dataset": dataset_name,
            **original_metrics,
        }
    )

    original_prediction_frames.append(
        original_predictions_df
    )

    print(
        f"✅ {dataset_name}: "
        f"max Δprob={max_probability_delta:.3e}, "
        f"max Δmetric={max_metric_delta:.3e}"
    )

identity_gate_df = pd.DataFrame(
    identity_gate_rows
)

identity_gate_df.to_csv(
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_temporal_order_original_identity_gate_seed{RUN_SEED}.csv",
    index=False,
)

direct_metrics_df = pd.DataFrame(
    direct_metric_rows
)

direct_predictions_all_df = pd.concat(
    direct_prediction_frames,
    ignore_index=True,
)

direct_metrics_df.to_csv(
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_direct_T8_metrics_seed{RUN_SEED}.csv",
    index=False,
)

direct_predictions_all_df.to_csv(
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_direct_T8_predictions_seed{RUN_SEED}.csv",
    index=False,
)

display(
    identity_gate_df
)

print(
    "\n✅ Semua Direct-T8 vs Original identity gates PASS"
)

# ============================================================
# OPTIONAL SAVED-RWF REPRODUCTION — source-notebook style
# ============================================================
saved_rwf_check_df = pd.DataFrame()

if SAVED_TEST_METRICS_PATH.exists():
    saved_metrics = pd.read_csv(
        SAVED_TEST_METRICS_PATH
    )

    if (
        "dataset"
        in saved_metrics.columns
        and (
            saved_metrics[
                "dataset"
            ]
            == "RWF"
        ).any()
    ):
        current_rwf = pd.DataFrame(
            original_metric_rows
        )

        current_rwf = current_rwf[
            current_rwf[
                "dataset"
            ]
            == "RWF"
        ].copy()

        saved_rwf = saved_metrics[
            saved_metrics[
                "dataset"
            ]
            == "RWF"
        ].copy()

        available_metrics = [
            metric_name
            for metric_name in [
                "accuracy",
                "f1_macro",
                "roc_auc",
            ]
            if (
                metric_name
                in current_rwf.columns
                and metric_name
                in saved_rwf.columns
            )
        ]

        if available_metrics:
            saved_rwf_check_df = current_rwf[
                [
                    "dataset",
                    *available_metrics,
                ]
            ].merge(
                saved_rwf[
                    [
                        "dataset",
                        *available_metrics,
                    ]
                ],
                on="dataset",
                how="inner",
                suffixes=(
                    "_retest",
                    "_saved",
                ),
            )

            delta_columns = []

            for metric_name in available_metrics:
                delta_column = (
                    f"delta_{metric_name}"
                )

                saved_rwf_check_df[
                    delta_column
                ] = (
                    saved_rwf_check_df[
                        f"{metric_name}_retest"
                    ]
                    -
                    saved_rwf_check_df[
                        f"{metric_name}_saved"
                    ]
                )

                delta_columns.append(
                    delta_column
                )

            saved_rwf_check_df[
                "max_abs_metric_delta"
            ] = saved_rwf_check_df[
                delta_columns
            ].abs().max(
                axis=1
            )

            max_saved_delta = float(
                saved_rwf_check_df[
                    "max_abs_metric_delta"
                ].max()
            )

            saved_rwf_check_df.to_csv(
                EVAL_OUTPUT_DIR
                / f"mobilenettsm_original_vs_saved_RWF_seed{RUN_SEED}.csv",
                index=False,
            )

            display(
                saved_rwf_check_df.round(
                    8
                )
            )

            if (
                max_saved_delta
                > MAX_ALLOWED_SAVED_RWF_DELTA
            ):
                message = (
                    f"RWF Original retest differs from saved evaluation "
                    f"by max={max_saved_delta:.6f}, exceeding "
                    f"{MAX_ALLOWED_SAVED_RWF_DELTA:.6f}."
                )

                if STRICT_SAVED_RWF_MATCH:
                    raise RuntimeError(
                        message
                    )

                print(
                    "⚠️",
                    message,
                )

            else:
                print(
                    "✅ Saved-RWF reproduction PASS."
                )

else:
    print(
        "ℹ️ Saved checkpoint test_metrics.csv tidak ditemukan; "
        "Direct-T8 vs Original identity gates tetap digunakan."
    )


ORIGINAL IDENTITY GATE | RWF


Direct T8:   0%|          | 0/250 [00:00<?, ?it/s]

Temporal order:   0%|          | 0/250 [00:00<?, ?it/s]

✅ RWF: max Δprob=0.000e+00, max Δmetric=0.000e+00

ORIGINAL IDENTITY GATE | UCF


Direct T8:   0%|          | 0/13 [00:00<?, ?it/s]

...

✅ BV: max Δprob=0.000e+00, max Δmetric=0.000e+00


,checkpoint_seed,dataset,max_abs_probability_delta,max_abs_metric_delta,delta_accuracy,delta_balanced_accuracy,delta_f1_macro,delta_roc_auc,delta_average_precision
0,0,RWF,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,UCF,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,BV,0.0,0.0,0.0,0.0,0.0,0.0,0.0



✅ Semua Direct-T8 vs Original identity gates PASS


,dataset,accuracy_retest,f1_macro_retest,roc_auc_retest,accuracy_saved,f1_macro_saved,roc_auc_saved,delta_accuracy,delta_f1_macro,delta_roc_auc,max_abs_metric_delta
0,RWF,0.669169,0.668918,0.722735,0.669169,0.668918,0.722735,0.0,0.0,0.0,0.0


✅ Saved-RWF reproduction PASS.


## 8. Run Reverse + 10× Shuffle-2/4/8

In [12]:
all_metric_frames = [
    pd.DataFrame(
        original_metric_rows
    )
]

all_prediction_frames = [
    pd.concat(
        original_prediction_frames,
        ignore_index=True,
    )
]

# ============================================================
# REVERSE
# ============================================================
if RUN_REVERSE:
    reverse_metric_rows = []

    reverse_prediction_frames = []

    for (
        dataset_name,
        store,
    ) in heldout_stores.items():
        (
            metrics,
            condition_predictions_df,
        ) = evaluate_order_condition(
            dataset_name=dataset_name,
            store=store,
            mode="reverse",
            repeat_id=0,
            chunk_size=None,
        )

        reverse_metric_rows.append(
            {
                "checkpoint_seed": RUN_SEED,
                "scenario": "reverse",
                "mode": "reverse",
                "chunk_size": np.nan,
                "repeat": 0,
                "dataset": dataset_name,
                **metrics,
            }
        )

        reverse_prediction_frames.append(
            condition_predictions_df
        )

    reverse_metrics_df = pd.DataFrame(
        reverse_metric_rows
    )

    reverse_predictions_df = pd.concat(
        reverse_prediction_frames,
        ignore_index=True,
    )

    all_metric_frames.append(
        reverse_metrics_df
    )

    all_prediction_frames.append(
        reverse_predictions_df
    )

    reverse_metrics_df.to_csv(
        EVAL_OUTPUT_DIR
        / f"metrics_reverse_seed{RUN_SEED}.csv",
        index=False,
    )

    reverse_predictions_df.to_csv(
        EVAL_OUTPUT_DIR
        / f"predictions_reverse_seed{RUN_SEED}.csv",
        index=False,
    )

# ============================================================
# SHUFFLES
# ============================================================
for chunk_size in SHUFFLE_CHUNK_SIZES:
    scenario = (
        f"shuffle_{chunk_size}"
    )

    for repeat_id in range(
        NUM_RANDOM_REPEATS
    ):
        print(
            f"\nseed={RUN_SEED} | "
            f"{scenario} | repeat={repeat_id}"
        )

        condition_metric_rows = []

        condition_prediction_frames = []

        for (
            dataset_name,
            store,
        ) in heldout_stores.items():
            (
                metrics,
                condition_predictions_df,
            ) = evaluate_order_condition(
                dataset_name=dataset_name,
                store=store,
                mode="shuffle",
                repeat_id=repeat_id,
                chunk_size=chunk_size,
            )

            condition_metric_rows.append(
                {
                    "checkpoint_seed": RUN_SEED,
                    "scenario": scenario,
                    "mode": "shuffle",
                    "chunk_size": chunk_size,
                    "repeat": repeat_id,
                    "dataset": dataset_name,
                    **metrics,
                }
            )

            condition_prediction_frames.append(
                condition_predictions_df
            )

        condition_metrics_df = pd.DataFrame(
            condition_metric_rows
        )

        condition_predictions_all_df = pd.concat(
            condition_prediction_frames,
            ignore_index=True,
        )

        all_metric_frames.append(
            condition_metrics_df
        )

        all_prediction_frames.append(
            condition_predictions_all_df
        )

        condition_metrics_df.to_csv(
            EVAL_OUTPUT_DIR
            / (
                f"metrics_{scenario}_"
                f"repeat{repeat_id:02d}_seed{RUN_SEED}.csv"
            ),
            index=False,
        )

        condition_predictions_all_df.to_csv(
            EVAL_OUTPUT_DIR
            / (
                f"predictions_{scenario}_"
                f"repeat{repeat_id:02d}_seed{RUN_SEED}.csv"
            ),
            index=False,
        )

print(
    "\n✅ Semua MobileNetV2+TSM held-out temporal-order conditions selesai."
)

Temporal order:   0%|          | 0/250 [00:00<?, ?it/s]

Temporal order:   0%|          | 0/13 [00:00<?, ?it/s]

Temporal order:   0%|          | 0/175 [00:00<?, ?it/s]


seed=0 | shuffle_2 | repeat=0


Temporal order:   0%|          | 0/250 [00:00<?, ?it/s]

...


seed=0 | shuffle_8 | repeat=9


Temporal order:   0%|          | 0/250 [00:00<?, ?it/s]

Temporal order:   0%|          | 0/13 [00:00<?, ?it/s]

Temporal order:   0%|          | 0/175 [00:00<?, ?it/s]


✅ Semua MobileNetV2+TSM held-out temporal-order conditions selesai.


## 9. Summary mean ± sampling std dan Δ terhadap Original

Definisi:

\[
\Delta M =
M_{\mathrm{Original}} -
M_{\mathrm{Perturbed}}
\]

Nilai positif berarti metric turun setelah urutan temporal diubah.

Untuk shuffle, `std_sampling` berasal dari **10 temporal permutations** pada
checkpoint seed yang sama; bukan standard deviation antar-training-seed.

AP adalah metric utama.

In [13]:
all_metrics = pd.concat(
    all_metric_frames,
    ignore_index=True,
)

all_predictions = pd.concat(
    all_prediction_frames,
    ignore_index=True,
)

all_metrics_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_temporal_order_metrics_all_repeats_seed{RUN_SEED}.csv"
)

all_predictions_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_temporal_order_predictions_all_repeats_seed{RUN_SEED}.csv"
)

all_metrics.to_csv(
    all_metrics_path,
    index=False,
)

all_predictions.to_csv(
    all_predictions_path,
    index=False,
)

METRIC_COLUMNS = [
    "accuracy",
    "balanced_accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "roc_auc",
    "average_precision",
    "precision_nonviolence",
    "recall_nonviolence",
    "f1_nonviolence",
    "precision_violence",
    "recall_violence",
    "f1_violence",
]

summary_rows = []

for (
    dataset_name,
    scenario,
), group in all_metrics.groupby(
    [
        "dataset",
        "scenario",
    ]
):
    row = {
        "checkpoint_seed": RUN_SEED,
        "dataset": dataset_name,
        "scenario": scenario,
        "num_repeats": int(
            len(group)
        ),
    }

    for metric_name in METRIC_COLUMNS:
        values = group[
            metric_name
        ].astype(float)

        row[
            f"{metric_name}_mean"
        ] = float(
            values.mean()
        )

        row[
            f"{metric_name}_std_sampling"
        ] = float(
            values.std(
                ddof=1
            )
            if len(values) > 1
            else 0.0
        )

        row[
            f"{metric_name}_min"
        ] = float(
            values.min()
        )

        row[
            f"{metric_name}_max"
        ] = float(
            values.max()
        )

    summary_rows.append(
        row
    )

summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    [
        "dataset",
        "scenario",
    ]
)

summary_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_temporal_order_summary_seed{RUN_SEED}.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

# ============================================================
# DELTA PER REPEAT
# ============================================================
original_lookup = (
    all_metrics[
        all_metrics[
            "scenario"
        ]
        == "original"
    ]
    .set_index(
        "dataset"
    )
)

delta_rows = []

for _, row in all_metrics.iterrows():
    if row[
        "scenario"
    ] == "original":
        continue

    dataset_name = row[
        "dataset"
    ]

    original_row = original_lookup.loc[
        dataset_name
    ]

    delta_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "scenario": row[
                "scenario"
            ],
            "repeat": int(
                row[
                    "repeat"
                ]
            ),

            "delta_accuracy_orig_minus_perturbed": float(
                original_row[
                    "accuracy"
                ]
                - row[
                    "accuracy"
                ]
            ),

            "delta_balanced_accuracy_orig_minus_perturbed": float(
                original_row[
                    "balanced_accuracy"
                ]
                - row[
                    "balanced_accuracy"
                ]
            ),

            "delta_f1_macro_orig_minus_perturbed": float(
                original_row[
                    "f1_macro"
                ]
                - row[
                    "f1_macro"
                ]
            ),

            "delta_auc_orig_minus_perturbed": float(
                original_row[
                    "roc_auc"
                ]
                - row[
                    "roc_auc"
                ]
            ),

            "delta_ap_orig_minus_perturbed": float(
                original_row[
                    "average_precision"
                ]
                - row[
                    "average_precision"
                ]
            ),

            "delta_f1_pp": float(
                100.0
                * (
                    original_row[
                        "f1_macro"
                    ]
                    - row[
                        "f1_macro"
                    ]
                )
            ),

            "delta_auc_pp": float(
                100.0
                * (
                    original_row[
                        "roc_auc"
                    ]
                    - row[
                        "roc_auc"
                    ]
                )
            ),

            "delta_ap_pp": float(
                100.0
                * (
                    original_row[
                        "average_precision"
                    ]
                    - row[
                        "average_precision"
                    ]
                )
            ),
        }
    )

delta_df = pd.DataFrame(
    delta_rows
)

delta_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_temporal_order_delta_all_repeats_seed{RUN_SEED}.csv"
)

delta_df.to_csv(
    delta_path,
    index=False,
)

delta_summary_rows = []

for (
    dataset_name,
    scenario,
), group in delta_df.groupby(
    [
        "dataset",
        "scenario",
    ]
):
    row = {
        "checkpoint_seed": RUN_SEED,
        "dataset": dataset_name,
        "scenario": scenario,
        "num_repeats": int(
            len(group)
        ),
    }

    for metric_name in [
        "delta_accuracy_orig_minus_perturbed",
        "delta_balanced_accuracy_orig_minus_perturbed",
        "delta_f1_macro_orig_minus_perturbed",
        "delta_auc_orig_minus_perturbed",
        "delta_ap_orig_minus_perturbed",
        "delta_f1_pp",
        "delta_auc_pp",
        "delta_ap_pp",
    ]:
        values = group[
            metric_name
        ].astype(float)

        row[
            f"{metric_name}_mean"
        ] = float(
            values.mean()
        )

        row[
            f"{metric_name}_std_sampling"
        ] = float(
            values.std(
                ddof=1
            )
            if len(values) > 1
            else 0.0
        )

        row[
            f"{metric_name}_min"
        ] = float(
            values.min()
        )

        row[
            f"{metric_name}_max"
        ] = float(
            values.max()
        )

    delta_summary_rows.append(
        row
    )

delta_summary_df = pd.DataFrame(
    delta_summary_rows
).sort_values(
    [
        "dataset",
        "scenario",
    ]
)

delta_summary_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_temporal_order_delta_summary_seed{RUN_SEED}.csv"
)

delta_summary_df.to_csv(
    delta_summary_path,
    index=False,
)

print(
    "=== TEMPORAL-ORDER SUMMARY ==="
)

display(
    summary_df[
        [
            "dataset",
            "scenario",
            "num_repeats",
            "average_precision_mean",
            "average_precision_std_sampling",
            "roc_auc_mean",
            "roc_auc_std_sampling",
            "f1_macro_mean",
            "f1_macro_std_sampling",
        ]
    ].round(
        5
    )
)

print(
    "=== DELTA VS ORIGINAL — POIN PERSENTASE ==="
)

display(
    delta_summary_df[
        [
            "dataset",
            "scenario",
            "num_repeats",
            "delta_ap_pp_mean",
            "delta_ap_pp_std_sampling",
            "delta_auc_pp_mean",
            "delta_f1_pp_mean",
        ]
    ].round(
        4
    )
)

print("Saved metrics      :", all_metrics_path)
print("Saved predictions  :", all_predictions_path)
print("Saved summary      :", summary_path)
print("Saved delta        :", delta_path)
print("Saved delta summary:", delta_summary_path)

=== TEMPORAL-ORDER SUMMARY ===


,dataset,scenario,num_repeats,average_precision_mean,average_precision_std_sampling,roc_auc_mean,roc_auc_std_sampling,f1_macro_mean,f1_macro_std_sampling
0,BV,original,1,0.65323,0.00000,0.66423,0.00000,0.43304,0.00000
1,BV,reverse,1,0.66184,0.00000,0.67213,0.00000,0.43435,0.00000
2,BV,shuffle_2,10,0.65449,0.00273,0.66170,0.00210,0.45490,0.00435
3,BV,shuffle_4,10,0.65320,0.00315,0.66301,0.00215,0.46714,0.00509
4,BV,shuffle_8,10,0.64974,0.00451,0.66288,0.00262,0.49152,0.00751
5,RWF,original,1,0.72905,0.00000,0.72274,0.00000,0.66892,0.00000
6,RWF,reverse,1,0.72432,0.00000,0.72059,0.00000,0.66096,0.00000
7,RWF,shuffle_2,10,0.72689,0.00204,0.72333,0.00152,0.67007,0.00341
8,RWF,shuffle_4,10,0.72505,0.00390,0.72481,0.00185,0.66788,0.00357
9,RWF,shuffle_8,10,0.72505,0.00223,0.72691,0.00131,0.66514,0.00203


=== DELTA VS ORIGINAL — POIN PERSENTASE ===


,dataset,scenario,num_repeats,delta_ap_pp_mean,delta_ap_pp_std_sampling,delta_auc_pp_mean,delta_f1_pp_mean
0,BV,reverse,1,-0.8609,0.0000,-0.7900,-0.1306
1,BV,shuffle_2,10,-0.1260,0.2729,0.2533,-2.1852
2,BV,shuffle_4,10,0.0032,0.3152,0.1219,-3.4100
3,BV,shuffle_8,10,0.3491,0.4508,0.1346,-5.8478
4,RWF,reverse,1,0.4732,0.0000,0.2149,0.7961
5,RWF,shuffle_2,10,0.2157,0.2036,-0.0595,-0.1147
6,RWF,shuffle_4,10,0.3998,0.3900,-0.2079,0.1037
7,RWF,shuffle_8,10,0.3998,0.2232,-0.4174,0.3779
8,UCF,reverse,1,-0.0184,0.0000,0.4600,-0.0905
9,UCF,shuffle_2,10,-0.0754,1.0052,0.0900,1.5021


Saved metrics      : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_TemporalOrder/seed_0/mobilenettsm_temporal_order_metrics_all_repeats_seed0.csv
Saved predictions  : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_TemporalOrder/seed_0/mobilenettsm_temporal_order_predictions_all_repeats_seed0.csv
Saved summary      : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_TemporalOrder/seed_0/mobilenettsm_temporal_order_summary_seed0.csv
Saved delta        : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_TemporalOrder/seed_0/mobilenettsm_temporal_order_delta_all_repeats_seed0.csv
Saved delta summary: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_TemporalOrder/seed_0/mobilenettsm_temporal_order_delta_summary_seed0.csv


## 10. Protocol manifest

In [14]:
protocol_audit = {
    "task": "heldout_temporal_order_evaluation_only",
    "source_protocol_notebook": (
        "Seed123_MobileNetTSM_TemporalOrder_TestOnly(1).ipynb"
    ),
    "model": "Zhang2022 MobileNetV2+TSM comparator",
    "model_variant": MODEL_VARIANT,
    "checkpoint_seed": RUN_SEED,
    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),
    "checkpoint_epoch": checkpoint.get(
        "epoch"
    ),
    "training_performed": False,
    "fine_tuning_performed": False,
    "optimizer_created": False,
    "strict_checkpoint_loading": True,
    "model_eval": (
        not model.training
    ),
    "trainable_parameters": 0,
    "cache_sequence_length": CACHE_SEQUENCE_LENGTH,
    "num_model_segments": NUM_SEGMENTS,
    "test_segment_indices_0based": (
        TEST_SEGMENT_INDICES.tolist()
    ),
    "perturbation_applied_after_segment_selection": True,
    "frame_content_preserved": True,
    "shuffle_chunk_sizes": SHUFFLE_CHUNK_SIZES,
    "relative_chunk_scales": {
        "shuffle_2": "25% of T=8 sequence",
        "shuffle_4": "50% of T=8 sequence",
        "shuffle_8": "100% of T=8 sequence",
    },
    "num_random_repeats": NUM_RANDOM_REPEATS,
    "permutation_base_seed": PERMUTATION_BASE_SEED,
    "checkpoint_seed_excluded_from_permutation_generation": True,
    "permutation_identity_inputs": [
        "permutation_base_seed",
        "repeat_id",
        "dataset_name_crc32",
        "sample_index",
        "chunk_size",
    ],
    "run_reverse": RUN_REVERSE,
    "tsm_shift_div": SHIFT_DIV,
    "tsm_blocks": model.tsm_blocks,
    "consensus": "mean_of_per_segment_logits",
    "use_amp": USE_AMP,
    "direct_original_identity_gate": True,
    "optional_saved_rwf_reproduction": True,
    "saved_rwf_match_tolerance": MAX_ALLOWED_SAVED_RWF_DELTA,
    "primary_metric": "average_precision",
    "delta_definition": (
        "Original minus Perturbed; positive means metric decreased "
        "after temporal-order perturbation."
    ),
    "heldout_datasets": list(
        HELDOUT_SPECS.keys()
    ),
    "no_cross_dataset_pooling": True,
    "checkpoint_config": checkpoint_config,
}

manifest_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_temporal_order_protocol_seed{RUN_SEED}.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol_audit,
        file,
        indent=2,
        default=str,
    )

print(
    json.dumps(
        protocol_audit,
        indent=2,
        default=str,
    )
)

print(
    "\nOutput:",
    EVAL_OUTPUT_DIR,
)

{
  "task": "heldout_temporal_order_evaluation_only",
  "source_protocol_notebook": "Seed123_MobileNetTSM_TemporalOrder_TestOnly(1).ipynb",
  "model": "Zhang2022 MobileNetV2+TSM comparator",
  "model_variant": "mobilenet_tsm",
  "checkpoint_seed": 0,
  "checkpoint_path": "/content/drive/MyDrive/IVS/ViolenceDetection/Models/Comparators/mobilenet_tsm_Seed0_v1/mobilenet_tsm/seed_0/best.pt",
  "checkpoint_epoch": 63,
  "training_performed": false,
  "fine_tuning_performed": false,
  "optimizer_created": false,
  "strict_checkpoint_loading": true,
  "model_eval": true,
  "trainable_parameters": 0,
  "cache_sequence_length": 16,
  "num_model_segments": 8,
  "test_segment_indices_0based": [
    1,
    3,
    5,
    7,
    9,
    11,
    13,
    15
  ],
  "perturbation_applied_after_segment_selection": true,
  "frame_content_preserved": true,
  "shuffle_chunk_sizes": [
    2,
    4,
    8
  ],
  "relative_chunk_scales": {
    "shuffle_2": "25% of T=8 sequence",
    "shuffle_4": "50% of T=8 seq

# Seed berikutnya

Setelah seed pertama selesai, **cukup ubah**:

```python
RUN_SEED = 0
```

menjadi:

```python
RUN_SEED = 42
```

kemudian:

```python
RUN_SEED = 123
```

Jangan ubah:

```python
CACHE_SEQUENCE_LENGTH = 16
NUM_SEGMENTS = 8
TEST_SEGMENT_INDICES = [1,3,5,7,9,11,13,15]

SHIFT_DIV = 8
EXPECTED_TSM_BLOCKS = 10

SHUFFLE_CHUNK_SIZES = [2,4,8]
NUM_RANDOM_REPEATS = 10
PERMUTATION_BASE_SEED = 20260810

BATCH_SIZE = 8
USE_AMP = False
```

Output:

```text
HeldOut_Evaluation/
└── MobileNetV2_TSM_TemporalOrder/
    ├── seed_0/
    ├── seed_42/
    └── seed_123/
```

File utama:
- `mobilenettsm_temporal_order_original_identity_gate_seedX.csv`
- `mobilenettsm_original_vs_saved_RWF_seedX.csv` *(jika saved metrics tersedia)*
- `mobilenettsm_direct_T8_metrics_seedX.csv`
- `mobilenettsm_direct_T8_predictions_seedX.csv`
- `metrics_reverse_seedX.csv`
- `predictions_reverse_seedX.csv`
- `mobilenettsm_temporal_order_metrics_all_repeats_seedX.csv`
- `mobilenettsm_temporal_order_predictions_all_repeats_seedX.csv`
- `mobilenettsm_temporal_order_summary_seedX.csv`
- `mobilenettsm_temporal_order_delta_all_repeats_seedX.csv`
- `mobilenettsm_temporal_order_delta_summary_seedX.csv`
- `temporal_order_permutation_audit_examples.json`
- `heldout_dataset_audit.csv`
- `mobilenettsm_temporal_order_protocol_seedX.json`